## Setup — find the repo root and point at the local model files

In [1]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the repo root (no pyproject.toml above this notebook)")

ROOT = find_repo_root(Path.cwd())
print("repo root:", ROOT)

# The Windows-level MODEL_* variables (if set) point at the old D: drive layout and
# make settings refuse to load. These three override them for this notebook only.
os.environ["MODEL_STORAGE_ROOT"] = str(ROOT / "data" / "runtime")
os.environ["MODEL_ARTIFACT_ROOT"] = str(ROOT / "data" / "runtime" / "models")
os.environ["MODEL_REGISTRY_PATH"] = str(ROOT / "data" / "runtime" / "manifests" / "model-registry.json")

sys.path.insert(0, str(ROOT))

repo root: C:\Users\wikiepeidia\OneDrive - caugiay.edu.vn\bài tập\usth\GEN14\INTERNSHIP\Internship-project


## The four messages

4 inputs

In [2]:
MESSAGES = {
    "bank_impersonation": (
        "Vietcombank: Tai khoan cua ban da bi tam khoa do phat hien giao dich bat thuong. "
        "Vui long dang nhap vcb-xacminh.vn de mo khoa ngay hom nay."
    ),
    "zalo_social_engineering": (
        "Chi oi, em la ban cua chi Lan, em dang can gap 2 trieu de dong vien phi nhap vien, "
        "chi chuyen giup em qua so tai khoan nay nhe, em gui lai ngay khi on."
    ),
    "task_scam": (
        "Viec nhe luong cao: chi can like va share bai viet, hoa hong 100k moi nhiem vu, "
        "dong phi tham gia 500k de bat dau nhan viec."
    ),
    "benign": (
        "Techcombank: Giao dich thanh cong! Tai khoan cua ban vua nhan duoc 2,000,000 VND "
        "tu NGUYEN VAN A luc 09:15. So du hien tai: 5,340,000 VND."
    ),
}
for label, text in MESSAGES.items():
    print(f"{label:24s} {text}")

bank_impersonation       Vietcombank: Tai khoan cua ban da bi tam khoa do phat hien giao dich bat thuong. Vui long dang nhap vcb-xacminh.vn de mo khoa ngay hom nay.
zalo_social_engineering  Chi oi, em la ban cua chi Lan, em dang can gap 2 trieu de dong vien phi nhap vien, chi chuyen giup em qua so tai khoan nay nhe, em gui lai ngay khi on.
task_scam                Viec nhe luong cao: chi can like va share bai viet, hoa hong 100k moi nhiem vu, dong phi tham gia 500k de bat dau nhan viec.
benign                   Techcombank: Giao dich thanh cong! Tai khoan cua ban vua nhan duoc 2,000,000 VND tu NGUYEN VAN A luc 09:15. So du hien tai: 5,340,000 VND.


## Part 1 — Qwen Model

.

In [3]:
from src.runtime.service import build_default_runtime_service

service = build_default_runtime_service()
print("backend ready, model loaded")

backend ready, model loaded


In [4]:
print(f"{'true class':<26}{'risk tier':<12}{'labels returned'}")
qwen_results = {}
for true_label, text in MESSAGES.items():
    result = service.analyze_text(text, channel="sms")
    qwen_results[true_label] = result
    print(f"{true_label:<26}{result.risk_tier:<12}{result.threat_labels}")

true class                risk tier   labels returned


llama_context: n_ctx_seq (512) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


bank_impersonation        high-risk   ['bank_impersonation']


zalo_social_engineering   high-risk   ['zalo_social_engineering', 'task_scam']


task_scam                 high-risk   ['task_scam', 'zalo_social_engineering']


benign                    benign      ['benign']


Full JSON Output

In [5]:
import json

sample = qwen_results["bank_impersonation"]
print(json.dumps(sample.model_dump(), indent=2, ensure_ascii=False))

{
  "risk_tier": "high-risk",
  "summary": "Structured decision from sms local runtime.",
  "top_cues": [
    {
      "span": "vcb-xacminh.vn",
      "reason": "Structured decision from sms local runtime.",
      "cue_type": "generic"
    },
    {
      "span": "Tai khoan cua ban da bi tam khoa do phat hien giao dich bat thuong",
      "reason": "Structured decision from sms local runtime.",
      "cue_type": "generic"
    },
    {
      "span": "dang nhap vcb-xacminh.vn de mo khoa",
      "reason": "Structured decision from sms local runtime.",
      "cue_type": "generic"
    }
  ],
  "threat_labels": [
    "bank_impersonation"
  ],
  "recommendations": [
    "Không truy cập các liên kết từ tin nhắn không rõ nguồn gốc.",
    "Kiểm tra lại lịch sử giao dịch trên ứng dụng ngân hàng chính thức.",
    "Liên hệ trực tiếp với Vietcombank qua số hotline chính thức để xác minh thông tin."
  ],
  "backend_name": "gguf",
  "provisional": true,
  "normalized_text": "Vietcombank: Tai khoan cua ba

## Part 2 — PhoBERT Model

In [6]:
import torch
import underthesea
from transformers import AutoModelForSequenceClassification, AutoTokenizer

PHOBERT_DIR = ROOT / "data" / "models" / "phase40" / "full" / "phobert" / "adapter-or-model"
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_DIR)
phobert = AutoModelForSequenceClassification.from_pretrained(PHOBERT_DIR).eval()
print("loaded. label positions:", phobert.config.id2label)

C:\Users\wikiepeidia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14741.82it/s]

loaded. label positions: {0: 'bank_impersonation', 1: 'zalo_social_engineering', 2: 'task_scam', 3: 'benign'}


In [7]:
for true_label, text in MESSAGES.items():
    # underthesea word-segments Vietnamese before the tokenizer sees it -- this is
    # the exact preprocessing step PhoBERT was trained and evaluated with.
    segmented = underthesea.word_tokenize(text, format="text")
    encoded = tokenizer(
        segmented, add_special_tokens=True, truncation=True, max_length=256, return_tensors="pt",
    )
    with torch.no_grad():
        logits = phobert(**encoded).logits[0]
    pred_id = int(torch.argmax(logits))
    pred_label = phobert.config.id2label[pred_id]
    print(f"\ntrue class:      {true_label}")
    print(f"segmented text:  {segmented[:90]}...")
    n_tokens = encoded['input_ids'].shape[1]
    print(f"input_ids ({n_tokens} tokens, first 12 shown): {encoded['input_ids'][0][:12].tolist()}")
    print(f"raw logits:      {[round(x, 3) for x in logits.tolist()]}")
    print(f"predicted class: {pred_label}  ({'correct' if pred_label == true_label else 'WRONG'})")


true class:      bank_impersonation
segmented text:  Vietcombank : Tai_khoan cua ban da_bi tam_khoa do phat hien giao dich bat thuong . Vui lon...
input_ids (41 tokens, first 12 shown): [0, 6061, 27, 27591, 4229, 3831, 718, 8926, 4899, 18495, 2054, 91]
raw logits:      [1.694, -0.778, -1.837, 0.621]
predicted class: bank_impersonation  (correct)

true class:      zalo_social_engineering
segmented text:  Chi_oi , em la_ban cua chi_Lan , em dang can_gap 2 trieu de dong vien phi nhap_vien , chi ...
input_ids (51 tokens, first 12 shown): [0, 8470, 13373, 4, 193, 11366, 718, 3831, 13086, 2507, 4, 193]
raw logits:      [0.768, 0.781, -1.63, -0.372]
predicted class: zalo_social_engineering  (correct)

true class:      task_scam
segmented text:  Viec nhe luong cao : chi_can like va share bai viet , hoa hong 100 k moi nhiem_vu , dong_p...
input_ids (39 tokens, first 12 shown): [0, 3696, 20250, 42014, 3198, 2662, 84, 27, 13086, 3694, 6955, 4116]
raw logits:      [-0.147, -0.324, 1.32, -0.663]
p